# Kaggle 02 — W3 Synthetic SFT Data Generation

**CPU only** — niente GPU necessaria per questi step.

Step eseguiti qui:
1. **MainframeBench** (~7k esempi curati, MIT) — import diretto da HF, ~5 min
2. **Gemini gold** (~5k esempi hard) — drip da Gemini 2.5 Flash free tier, 1500 req/giorno
   → esegui questo notebook ogni giorno per 3-4 giorni fino a 5k esempi

Step NON eseguiti qui (richiedono GPU / Modal):
3. Teacher vLLM bulk (~35k) — Modal A100 con XMAiNframe-10.5b
4. DPO pairs (~4k) — Modal A100

**Prerequisiti Kaggle Secrets:** `HF_TOKEN`, `GEMINI_API_KEY`

Output: HF Hub private `AlexThunder0/cobol-sft-dataset`

In [ ]:
!pip install datasets huggingface-hub google-generativeai sqlitedict tqdm -q

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GEMINI_API_KEY'] = secrets.get_secret('GEMINI_API_KEY')

print('Secrets configurati')

In [ ]:
import subprocess, sys, os, shutil

GH_TOKEN = secrets.get_secret('GH_TOKEN')
shutil.rmtree('/kaggle/working/Qwen-COBOL', ignore_errors=True)

with open(os.path.expanduser('~/.netrc'), 'w') as f:
    f.write(f'machine github.com login x-access-token password {GH_TOKEN}\n')
os.chmod(os.path.expanduser('~/.netrc'), 0o600)

subprocess.run(
    ['git', 'clone', '--depth', '1', '--quiet',
     'https://github.com/AlexThunder01/Qwen-COBOL.git',
     '/kaggle/working/Qwen-COBOL'],
    check=True, env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'},
)
sys.path.insert(0, '/kaggle/working/Qwen-COBOL')
print('Repo clonato')

In [ ]:
# ── Step 1: MainframeBench (~7k esempi, MIT, CPU, ~5 min) ────────────────────
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

from src.synth.distill_orchestrator import step_mainframebench
step_mainframebench()
print('MainframeBench importato.')

In [ ]:
# ── Step 2: Gemini gold (1500 req/giorno, riprende da cache SQLite) ───────────
# Esegui questa cella ogni giorno. Accumula nella cache e pusha quando raggiunge 5k.
# Stima: 3-4 giorni per completare i 5k esempi.

from src.synth.distill_orchestrator import step_gemini
step_gemini()
print('Gemini gold done per oggi.')

In [ ]:
# ── Statistiche dataset SFT corrente ─────────────────────────────────────────
from datasets import load_dataset
import pandas as pd

splits_info = []
for split in ['mainframebench', 'gemini_gold']:
    try:
        ds = load_dataset('AlexThunder0/cobol-sft-dataset', split=split,
                         token=os.environ['HF_TOKEN'])
        splits_info.append({'split': split, 'esempi': len(ds)})
    except Exception as e:
        splits_info.append({'split': split, 'esempi': f'non trovato ({e})'})

df = pd.DataFrame(splits_info)
print('=== SFT DATASET STATS ===')
print(df.to_string(index=False))